In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import max
import time

# Stop existing Spark session if needed
SparkSession.builder.getOrCreate().stop()

# Start new Spark session
spark = SparkSession.builder\
        .master("spark://192.168.2.213:7077") \
        .appName("test_cluster")\
        .config("spark.driver.host", "192.168.2.107") \
        .config("spark.driver.port", "4041") \
        .config("spark.ui.port", "4042") \
        .config("spark.dynamicAllocation.enabled", True)\
        .config("spark.dynamicAllocation.shuffleTracking.enabled", True)\
        .config("spark.shuffle.service.enabled", False)\
        .config("spark.dynamicAllocation.executorIdleTimeout", "30s")\
        .config("spark.executor.cores", 2)\
        .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Load dataset
for i in range(0, 4):
    file_path = f'hdfs://192.168.2.213:9000/data/us-counties-202{i}.csv'
    df = spark.read.csv(file_path, header=True, inferSchema=True)
    
    start_time = time.time()
    
    max_deaths_value = df.select(max("deaths")).collect()[0][0]
    
    
    
    print(f"Max deaths: {max_deaths}")
    max_death_county = df.filter(col("deaths") == max_deaths_value).select("county", "state", "deaths","date")
    
    # Show the result
    max_death_county.show()
    end_time = time.time()
    execution_time = end_time - start_time
    print(f"Execution time: {execution_time:.4f} seconds")
